# omniASR on the 30 office-testing recordings

Transcribes the 30 Survey2026 office-testing recordings (Sep 17–21, 2026) with Meta's Omnilingual ASR (omniASR), for comparison with the whisper.cpp models in `CER_WER_calculation.ipynb`.

## Background

### What this notebook does
[Omnilingual ASR](https://github.com/facebookresearch/omnilingual-asr) is Meta's speech recognition model family covering 1,600+ languages, including Swahili (`swh_Latn`). PazaBench ranks the largest model, omniASR-LLM-7B, among the most accurate for Swahili. This notebook runs omniASR models on the same 30 recordings used in `CER_WER_calculation.ipynb` (the last take of Q8–Q17 from Victor, Susan and Ednah) and saves what each model heard.

The models are PyTorch / fairseq2 checkpoints, not whisper.cpp files, and the larger ones need a GPU, so this notebook runs on **Google Colab** rather than on the Mac.

| Model card | Size | Download | GPU memory | Notes |
|---|---|---|---|---|
| `omniASR_LLM_7B` | ~7.8B params | 31 GB | ~17 GiB | Most accurate; needs an A100 or L4 |
| `omniASR_LLM_300M`, `omniASR_LLM_300M_v2` | 1.6B (300M encoder + LLM decoder) | 6.1 GiB | ~5 GiB | Runs on any Colab GPU |
| `omniASR_CTC_300M`, `omniASR_CTC_300M_v2` | 325M | 1.3 GiB | ~2 GiB | Closest to Whisper Small in size; very fast |

### What you need
- A Google account with a Colab plan that offers an **A100** or **L4** GPU (Google AI Plus does). The free T4 (16 GB) is too small for the 7B model; the notebook falls back to the 3B model on it.
- No Hugging Face token: the omniASR models and the recordings are public.

### How to run it
1. Open [colab.research.google.com](https://colab.research.google.com) and upload this notebook (File → Upload notebook).
2. **Runtime → Change runtime type → A100 GPU** (or L4). Leave High-RAM off.
3. Choose the models in cell 1. By default it runs `omniASR_LLM_7B` on a 22 GB+ GPU and `omniASR_LLM_3B` otherwise. To run other models, set `MODEL_CARDS` by hand, e.g. `["omniASR_LLM_300M", "omniASR_LLM_300M_v2", "omniASR_CTC_300M", "omniASR_CTC_300M_v2"]`.
4. **Runtime → Run all.**
5. Cell 5 downloads `omniasr_transcripts.json`. Save it to `ASR_Accuracy/office_testing/` (rename it if your browser adds a number).

### What to expect
- **Cell 2 (install), 3–5 minutes.** omnilingual-asr needs fairseq2 0.6, which only works with PyTorch 2.8 on Python 3.10–3.12, so the notebook builds a separate Python 3.12 environment in `/content/omni` and leaves Colab's own Python untouched. Re-running it in the same session skips the environment that already exists.
- **Cell 4 (transcribe), first run.** Downloading a model is the slow part: about 20–25 minutes for the 31 GB 7B checkpoint ("loaded in ~1400 s"), a few minutes for the smaller ones. Disk usage in Colab's Resources panel climbs while it downloads. Once loaded, each recording takes about 1 second (LLM models) or a few hundredths of a second (CTC models).
- **Running more models later in the same session:** change `MODEL_CARDS` in cell 1 and re-run only cells 1, 4 and 5. Cells 2 and 3 don't need to run again. If cell 4 says `RECORDINGS` is not defined, the session was reset; run cell 3 first.
- **Audio limit:** these models only accept clips under 40 seconds. The longest recording here (Victor's Q15) is 39.3 s.

### Results so far (Oct 2026)
Scored against `office_testing/expected_answers.json` with numbers normalised to Swahili words (see the comparison spreadsheet `Swahili_Whisper_Model_Comparison.xlsx`):

| Model | WER | CER |
|---|---|---|
| omniASR_LLM_7B | 0.071 | 0.021 |
| omniASR_LLM_300M | 0.116 | 0.038 |
| omniASR_LLM_300M_v2 | 0.165 | 0.026 |
| omniASR_CTC_300M | 0.262 | 0.063 |
| omniASR_CTC_300M_v2 | 0.356 | 0.054 |

For comparison, the best whisper.cpp model, dmusingu CV14, scores 0.146 / 0.057. omniASR-LLM-7B is the most accurate overall but far too large for a phone.

## Example: one answer across engines

Ednah's answer to Q17 (survey `7db59ab6`, recording `audio/2026-09-21_7db59ab6_Ednah_Q17.wav`):

> **Expected:** Nauza kwa wafanyabiashara wa eneo kwa sababu wako karibu na hulipa haraka.<br>
> *I sell to local traders because they are nearby and pay quickly.*

| Engine | Ran on | Transcript | Word errors (of 12) | CER |
|---|---|---|---|---|
| OpenAI generic small (current model) | Phone | Na uza kwa fanya bieshara wa yene wa kwa sababu, wakoka ribuna ulipa araka. | 11 | 0.192 |
| OpenAI generic small (current model) | Mac, whisper.cpp | Nauza kwa fanya bieshara wa yene wa kwa sababu wa koka ribuna ulipa araka. | 9 | 0.192 |
| Lingua-Connect | Mac, whisper.cpp | nauza kwa wafanya biashara waineeo kwa sababu wa kukaribu na ulipa araka | 7 | 0.123 |
| CDLI CV20 | Mac, whisper.cpp | Na uza kwa wafanyabiashara wanaoneo kwa sababu wako karibu na ulipa haraka | 5 | 0.068 |
| CDLI Kenyan non-standard | Mac, whisper.cpp | Nauza kwa wafanyabiashara wale eneo kwa sababu wako karibu na ulipa haraka. | 2 | 0.041 |
| **dmusingu CV14** | Mac, whisper.cpp | nauza kwa wafanyabiashara wa eneo kwa sababu wako karibu na ulipa haraka | 1 | 0.014 |
| **omniASR LLM 7B** | Colab A100 | nauza kwa wafanyabiashara wa eneo kwa sababu wako karibu na hulipa haraka | 0 | 0.000 |
| omniASR LLM 300M | Colab A100 | nauza kwa wafanyabiashara wa eneo kwa sababu wako karibu na ulipa haraka | 1 | 0.014 |
| omniASR LLM 300M v2 | Colab A100 | nauza kwa wafanyabiashara wa eneo kwa sababu wako karibu na ulipa haraka | 1 | 0.014 |
| omniASR CTC 300M | Colab A100 | nauza kwa wafanya biashara waeneo kwa sababu wako karibu na ulipa araka | 5 | 0.055 |
| omniASR CTC 300M v2 | Colab A100 | nauza kwa wafanya biashara waeneo kwa sababu wakwa karibu na ulipa araka | 6 | 0.082 |

Word errors and CER are scored the same way as `CER_WER_calculation.ipynb`: lowercase, punctuation removed, numbers as Swahili words.

The current model breaks words apart and mishears the ends of the sentence ("fanya bieshara" for *wafanyabiashara*, "ribuna ulipa araka" for *karibu na hulipa haraka*). The Swahili fine-tunes fix most of that. CV14 and the omniASR LLM models miss only the "h" in *hulipa*, and omniASR LLM 7B gets the whole answer exactly right. The CTC models get the sounds right but split *wafanyabiashara* and drop the "h" in *haraka*.

In [ ]:
# 1. Check the GPU and pick the model
import torch
assert torch.cuda.is_available(), "No GPU - choose a GPU runtime first (Runtime > Change runtime type)"
gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 1024**3
print(f"GPU: {gpu.name}, {gpu_gb:.1f} GB")

# omniASR-LLM-7B needs ~17 GiB in bf16; leave headroom for activations.
# Set MODEL_CARDS by hand to override, e.g. ["omniASR_LLM_7B_v2"] or
# ["omniASR_LLM_300M", "omniASR_LLM_300M_v2", "omniASR_CTC_300M", "omniASR_CTC_300M_v2"]
MODEL_CARDS = ["omniASR_LLM_7B"] if gpu_gb >= 22 else ["omniASR_LLM_3B"]
print("Will run:", MODEL_CARDS)

In [ ]:
# 2. Install Omnilingual ASR in its own Python 3.12 environment (a few minutes)
# omnilingual-asr needs fairseq2 0.5.2-0.6.0, which only has builds for PyTorch 2.7.1/2.8.0 on Python 3.10-3.12.
# Colab's own Python/PyTorch are newer, so this uses a separate environment and leaves Colab's untouched.
!apt-get -qq install -y libsndfile1 > /dev/null
!pip -q install uv
import os
if not os.path.exists("/content/omni/bin/python"):
    !uv venv -q --python 3.12 /content/omni
!uv pip install -q --python /content/omni/bin/python --index-strategy unsafe-best-match "torch==2.8.0" "torchaudio==2.8.0" "fairseq2[arrow]==0.6.0" omnilingual-asr --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128
import subprocess
subprocess.run(["/content/omni/bin/python", "-c", "import torch, fairseq2, omnilingual_asr; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| fairseq2', fairseq2.__version__)"], check=True)

In [ ]:
# 3. Download the 30 recordings from GitHub (same entries as office_testing/expected_answers.json)
import json, os, urllib.request
RECORDINGS = [
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q8",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q8_20260917_124101.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q9",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q9_20260917_124248.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q10",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q10_20260917_124425.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q11",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q11_20260917_124543.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q12",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q12_20260917_125021.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q13",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q13_20260917_125119.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q14",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q14_20260917_125429.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q15",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q15_20260917_125546.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q16",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q16_20260917_125652.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q17",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q17_20260917_125950.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q8",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q8_20260921_113046.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q9",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q9_20260921_113118.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q10",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q10_20260921_113155.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q11",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q11_20260921_113235.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q12",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q12_20260921_113324.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q13",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q13_20260921_113403.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q14",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q14_20260921_113434.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q15",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q15_20260921_113516.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q16",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q16_20260921_113603.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q17",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q17_20260921_113640.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q8",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q8_20260921_053701.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q9",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q9_20260921_053808.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q10",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q10_20260921_054007.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q11",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q11_20260921_054045.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q12",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q12_20260921_054121.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q13",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q13_20260921_054145.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q14",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q14_20260921_054214.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q15",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q15_20260921_054249.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q16",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q16_20260921_054346.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q17",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q17_20260921_054455.wav"
 }
]
os.makedirs("wavs", exist_ok=True)
for r in RECORDINGS:
    r["path"] = os.path.join("wavs", r["url"].rsplit("/", 1)[-1])
    if not os.path.exists(r["path"]):
        urllib.request.urlretrieve(r["url"], r["path"])
print(f"{len(RECORDINGS)} recordings ready")

In [ ]:
# 4. Transcribe in Swahili (swh_Latn) using the omniASR environment, and save the results
SCRIPT = 'import gc, json, os, sys, time, torch\nfrom omnilingual_asr.models.inference.pipeline import ASRInferencePipeline\n\ncards, recordings = sys.argv[1].split(","), json.load(open("recordings.json"))\ngpu = torch.cuda.get_device_name(0)\nresults = {}\nfor card in cards:\n    t0 = time.time()\n    pipeline = ASRInferencePipeline(model_card=card)  # bf16 on the GPU by default\n    print(f"{card}: loaded in {time.time() - t0:.0f} s", flush=True)\n    transcripts = []\n    for r in recordings:\n        t1 = time.time()\n        text = pipeline.transcribe([r["path"]], lang=["swh_Latn"], batch_size=1)[0]\n        transcripts.append({"survey_id": r["survey_id"], "question_id": r["question_id"],\n                            "wav_file": os.path.basename(r["path"]), "transcript": text,\n                            "seconds": round(time.time() - t1, 1)})\n        print(f"  {r[\'survey_id\'][:8]} {r[\'question_id\']:>3}: {text}", flush=True)\n    results[card] = {"model_file": card, "transcripts": transcripts}\n    del pipeline; gc.collect(); torch.cuda.empty_cache()\n\nout = {"description": "omniASR transcripts of the 30 office_testing recordings (expected_answers.json).",\n       "runtime": f"omnilingual-asr on Google Colab, {gpu}, bf16, torch {torch.__version__}",\n       "decoding": "ASRInferencePipeline defaults, lang swh_Latn, batch_size 1",\n       "models": results}\njson.dump(out, open("omniasr_transcripts.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)\nprint("Saved omniasr_transcripts.json")\n'
open('transcribe.py', 'w').write(SCRIPT)
json.dump(RECORDINGS, open('recordings.json', 'w'))
cards = ','.join(MODEL_CARDS)
!/content/omni/bin/python transcribe.py {cards}

In [ ]:
# 5. Download the results
from google.colab import files
files.download("omniasr_transcripts.json")